In [ ]:
# Konfiguracja i importy
import os
import json
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import pathlib

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, label_binarize
)

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras import layers

# XGBoost
import xgboost as xgb

# Transformers
import transformers
import torch
from transformers import BertTokenizer

# Stałe
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)

# Ścieżki
DATA_DIR = pathlib.Path("../data")
REPORTS_DIR = pathlib.Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Klasy
CLASS_NAMES = ["non-toxic", "toxic", "severely-toxic"]
N_CLASSES = 3

print("Konfiguracja zakończona")

## 1. EDA - Eksploracyjna Analiza Danych

In [ ]:
# Wczytaj surowe dane
df_raw = pd.read_csv(DATA_DIR / "raw/train.csv")
print(f"Rozmiar surowych danych: {df_raw.shape}")
print(f"\nPrzykładowe wiersze:")
df_raw.head()

In [ ]:
# Statystyki etykiet
labels = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

print("Rozkład etykiet w surowych danych:")
for label in labels:
    count = df_raw[label].sum()
    pct = df_raw[label].mean() * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

# Łączna toksyczność
df_raw["is_toxic"] = df_raw[labels].max(axis=1)
toxic_count = df_raw["is_toxic"].sum()
print(f"\nŁącznie komentarzy toksycznych: {toxic_count} ({toxic_count/len(df_raw)*100:.1f}%)")

In [ ]:
# Przykładowe komentarze toksyczne
print("\nPrzykładowe komentarze toksyczne:")
toxic_samples = df_raw[df_raw["is_toxic"] == 1]["comment_text"].sample(3, random_state=SEED)
for i, text in enumerate(toxic_samples, 1):
    print(f"{i}. {text[:100]}...")

print("\nPrzykładowe komentarze nietoksyczne:")
nontoxic_samples = df_raw[df_raw["is_toxic"] == 0]["comment_text"].sample(3, random_state=SEED)
for i, text in enumerate(nontoxic_samples, 1):
    print(f"{i}. {text[:100]}...")

In [ ]:
# Analiza długości komentarzy
df_raw["text_length"] = df_raw["comment_text"].str.len()

print("Statystyki długości komentarzy:")
print(df_raw["text_length"].describe())

# Rozkład długości
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(df_raw["text_length"], bins=50, alpha=0.7)
ax1.set_title("Rozkład długości komentarzy")
ax1.set_xlabel("Długość tekstu")
ax1.set_ylabel("Liczba komentarzy")

ax2.boxplot([df_raw[df_raw["is_toxic"]==0]["text_length"], 
             df_raw[df_raw["is_toxic"]==1]["text_length"]], 
            labels=["Non-toxic", "Toxic"])
ax2.set_title("Porównanie długości")
ax2.set_ylabel("Długość tekstu")

plt.tight_layout()
plt.show()

In [ ]:
# Najczęstsze słowa
def clean_text(text):
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())
    return text

# Podział na toksyczne i nietoksyczne
toxic_texts = df_raw[df_raw["is_toxic"] == 1]["comment_text"].apply(clean_text)
nontoxic_texts = df_raw[df_raw["is_toxic"] == 0]["comment_text"].apply(clean_text)

# Najczęstsze słowa
toxic_words = " ".join(toxic_texts).split()
nontoxic_words = " ".join(nontoxic_texts).split()

toxic_freq = Counter(toxic_words).most_common(20)
nontoxic_freq = Counter(nontoxic_words).most_common(20)

# Wykresy
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Toxic words bar
words, counts = zip(*toxic_freq)
ax1.barh(words, counts)
ax1.set_title("Najczęstsze słowa - komentarze toksyczne")

# Non-toxic words bar
words, counts = zip(*nontoxic_freq)
ax2.barh(words, counts)
ax2.set_title("Najczęstsze słowa - komentarze nietoksyczne")

# Word clouds
toxic_text = " ".join(toxic_words[:10000])  # Limit for performance
wordcloud_toxic = WordCloud(width=400, height=300, background_color="white", 
                           max_words=100).generate(toxic_text)
ax3.imshow(wordcloud_toxic, interpolation="bilinear")
ax3.axis("off")
ax3.set_title("Chmura słów - toksyczne")

nontoxic_text = " ".join(nontoxic_words[:10000])
wordcloud_nontoxic = WordCloud(width=400, height=300, background_color="white",
                              max_words=100).generate(nontoxic_text)
ax4.imshow(wordcloud_nontoxic, interpolation="bilinear")
ax4.axis("off")
ax4.set_title("Chmura słów - nietoksyczne")

plt.tight_layout()
plt.show()

## 2. Przygotowanie Danych i Split

In [ ]:
# Wczytaj i oczyść dane
df_all = pd.read_csv(DATA_DIR / "raw/train.csv").dropna().drop_duplicates()

# Utwórz poziom toksyczności (3 klasy)
mild_toxic = df_all["toxic"].astype(bool) | df_all["obscene"].astype(bool) | df_all["insult"].astype(bool)
severe_toxic = df_all["severe_toxic"].astype(bool) | df_all["threat"].astype(bool) | df_all["identity_hate"].astype(bool)

df_all["toxic_level"] = 0
df_all.loc[mild_toxic, "toxic_level"] = 1
df_all.loc[severe_toxic, "toxic_level"] = 2

# Usuń stare kolumny
df_all = df_all.drop(["id", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"], axis=1)

print(f"Po przetworzeniu: {df_all.shape}")
print("Rozkład klas:")
print(df_all["toxic_level"].value_counts().sort_index())

In [ ]:
# Czyszczenie tekstu
import nltk
from nltk.corpus import stopwords

# Pobierz stopwords jeśli potrzebne
try:
    sw = stopwords.words("english")
except:
    nltk.download("stopwords")
    sw = stopwords.words("english")

def preprocess_text(text):
    # Usuń URL
    text = re.sub(r'https?://\S+', "", text)
    # Małe litery
    text = text.lower()
    # Usuń apostrofy
    text = re.sub(r"'", "", text)
    # Tylko litery i spacje
    text = re.sub(r"[^a-z]+", " ", text)
    # Usuń stopwords i ogranicz długość
    words = [w for w in text.split() if w not in sw][:1024]
    return " ".join(words).strip()

df_all["comment_text"] = df_all["comment_text"].apply(preprocess_text)
print("Tekst wyczyszczony")

In [ ]:
# Podział na train/val/test ze stratyfikacją
df_train_val, df_test = train_test_split(df_all, test_size=5000, stratify=df_all["toxic_level"], random_state=SEED)
df_train, df_val = train_test_split(df_train_val, test_size=2000, stratify=df_train_val["toxic_level"], random_state=SEED)

# Zapisz pliki
df_train.to_csv(DATA_DIR / "df_train.csv", index=False)
df_val.to_csv(DATA_DIR / "df_val.csv", index=False)
df_test.to_csv(DATA_DIR / "df_test.csv", index=False)

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print("\nRozkład klas w train:")
print(df_train["toxic_level"].value_counts().sort_index())
print("\nRozkład klas w val:")
print(df_val["toxic_level"].value_counts().sort_index())
print("\nRozkład klas w test:")
print(df_test["toxic_level"].value_counts().sort_index())

## 3. Przygotowanie Sample Testowego (500 próbek)

In [ ]:
# Stratyfikowany sample z testu
SAMPLE_SIZE = 500
df_test_sample, _ = train_test_split(
    df_test,
    train_size=SAMPLE_SIZE,
    stratify=df_test["toxic_level"],
    random_state=SEED
)

df_test_sample = df_test_sample.reset_index(drop=True)
df_test_sample['original_idx'] = df_test_sample.index

# Zapisz sample
df_test_sample.to_csv(DATA_DIR / "df_test_sample.csv", index=False)

# Zapisz indeksy dla BERT
sample_indices = df_test.index[df_test['comment_text'].isin(df_test_sample['comment_text'])].tolist()
with open(DATA_DIR / "sample_indices.txt", "w") as f:
    f.write(str(sample_indices))

print(f"Test sample: {len(df_test_sample)} próbek")
print("Rozkład klas:")
print(df_test_sample['toxic_level'].value_counts().sort_index())

## 4. Modele

In [ ]:
# Wczytaj dane treningowe
df_train = pd.read_csv(DATA_DIR / "df_train.csv")
df_val = pd.read_csv(DATA_DIR / "df_val.csv")
df_test_sample = pd.read_csv(DATA_DIR / "df_test_sample.csv")

# Przygotuj dane
X_train = df_train['comment_text'].fillna('')
y_train = df_train['toxic_level'].values

X_val = df_val['comment_text'].fillna('')
y_val = df_val['toxic_level'].values

X_test = df_test_sample['comment_text'].fillna('')
y_test = df_test_sample['toxic_level'].values

print(f"Dane przygotowane: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

### 4.1 Baseline: TF-IDF + Logistic Regression

In [ ]:
# Pipeline TF-IDF + LogReg
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=200_000,
)

clf_baseline = LogisticRegression(
    solver="lbfgs",
    class_weight="balanced",
    C=2.0,
    max_iter=1000,
    random_state=SEED,
    multi_class="multinomial",
)

pipe_baseline = Pipeline([("tfidf", tfidf), ("clf", clf_baseline)])
pipe_baseline.fit(X_train, y_train)

print("Baseline wytrenowany")

In [ ]:
# Predykcje baseline
y_pred_baseline = pipe_baseline.predict(X_test)
y_prob_baseline = pipe_baseline.predict_proba(X_test)

print("Predykcje baseline wykonane")
print(f"Unikalne predykcje: {np.unique(y_pred_baseline, return_counts=True)}")

### 4.2 XGBoost z Embeddings

In [ ]:
# TextVectorization layer
max_features = 100000
sequence_length = 1024

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)

# Adapt na danych treningowych
vectorize_layer.adapt(X_train.values)
print(f"Rozmiar słownika: {len(vectorize_layer.get_vocabulary())}")

In [ ]:
# Wektoryzacja danych
def vectorize_texts(texts):
    return vectorize_layer(texts).numpy()

X_train_vec = vectorize_texts(X_train.values)
X_test_vec = vectorize_texts(X_test.values)

print(f"Kształt X_train_vec: {X_train_vec.shape}")
print(f"Kształt X_test_vec: {X_test_vec.shape}")

In [ ]:
# Trenowanie XGBoost
dtrain = xgb.DMatrix(X_train_vec, label=y_train)
dtest = xgb.DMatrix(X_test_vec, label=y_test)

param = {
    'max_depth': 4,
    'eta': 1,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'nthread': 4,
    'seed': SEED
}

num_round = 25
bst = xgb.train(param, dtrain, num_round, verbose_eval=False)

print("XGBoost wytrenowany")

In [ ]:
# Predykcje XGBoost
y_prob_xgb = bst.predict(dtest)
y_pred_xgb = np.argmax(y_prob_xgb, axis=1)

print("Predykcje XGBoost wykonane")
print(f"Unikalne predykcje: {np.unique(y_pred_xgb, return_counts=True)}")

### 4.3 BERT (z zapisanych predykcji)

In [ ]:
# Wczytaj predykcje BERT (wersja 128 tokenów)
with open("BERTpred128.txt", "r") as f:
    bert_preds_full = eval(f.read())

# Wyciągnij predykcje dla naszego sample
with open(DATA_DIR / "sample_indices.txt", "r") as f:
    sample_indices = eval(f.read())

y_pred_bert = np.array([bert_preds_full[i] for i in sample_indices])

print(f"Predykcje BERT wyciągnięte: {len(y_pred_bert)}")
print(f"Unikalne predykcje: {np.unique(y_pred_bert, return_counts=True)}")

### 4.4 LLM: Groq API (Llama 3.1 70B)

**Uwaga:** Wymaga klucza API Groq. Ustaw zmienną środowiskową `GROQ_API_KEY`.

Darmowy tier: 6000 requestów/dzień, 30 req/min.

In [ ]:
# Konfiguracja LLM
LLM_MODEL = "llama-3.1-70b-versatile"
LLM_PREDICTIONS_FILE = DATA_DIR / "llm_predictions.json"

# Sprawdź zapisane predykcje
if LLM_PREDICTIONS_FILE.exists():
    with open(LLM_PREDICTIONS_FILE, "r") as f:
        llm_cache = json.load(f)
    print(f"Załadowano {len(llm_cache)} zapisanych predykcji LLM")
else:
    llm_cache = {}
    print("Brak zapisanych predykcji")

In [ ]:
# Funkcja do klasyfikacji przez LLM
def classify_with_llm(client, text, max_retries=3):
    """Klasyfikuje tekst używając LLM z temperature=0."""
    prompt = f"""Klasyfikuj następujący komentarz jako jedną z kategorii: 0 (nietoksyczny), 1 (toksyczny), lub 2 (bardzo toksyczny).

Zasady:
- 0 = nietoksyczny: Normalna, szanująca dyskusja
- 1 = toksyczny: Opryskliwy, nieuprzejmy lub lekko obraźliwy
- 2 = bardzo toksyczny: Mowa nienawiści, groźby, ekstremalne przekleństwa

Odpowiedz TYLKO pojedynczą cyfrą: 0, 1, lub 2.

Komentarz: {text[:1500]}

Klasyfikacja:"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=10,
            )
            answer = response.choices[0].message.content.strip()
            
            # Wyciągnij cyfrę
            match = re.search(r'[012]', answer)
            if match:
                return int(match.group())
            else:
                print(f"Nieprawidłowa odpowiedź: {answer}")
                return 0  # Domyślnie nietoksyczny
                
        except Exception as e:
            print(f"Próba {attempt+1} nieudana: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    
    return 0  # Domyślnie przy błędzie

In [ ]:
# Uruchom inferencję LLM (z rate limiting i cache)
try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False
    print("Groq nie zainstalowany. Uruchom: pip install groq")

if GROQ_AVAILABLE and os.environ.get("GROQ_API_KEY"):
    client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    
    y_pred_llm = []
    
    for i, (idx, text) in enumerate(zip(df_test_sample['original_idx'], X_test)):
        cache_key = str(idx)
        
        if cache_key in llm_cache:
            pred = llm_cache[cache_key]
        else:
            pred = classify_with_llm(client, text)
            llm_cache[cache_key] = pred
            
            # Rate limiting: 30 req/min
            if (i + 1) % 25 == 0:
                time.sleep(60)  # Czekaj 1 minutę co 25 requestów
                
            # Zapisz cache co jakiś czas
            if (i + 1) % 50 == 0:
                with open(LLM_PREDICTIONS_FILE, "w") as f:
                    json.dump(llm_cache, f)
                print(f"Przetworzono {i+1}/{len(X_test)} próbek")
        
        y_pred_llm.append(pred)
    
    # Końcowy zapis
    with open(LLM_PREDICTIONS_FILE, "w") as f:
        json.dump(llm_cache, f)
    
    y_pred_llm = np.array(y_pred_llm)
    print(f"\nPredykcje LLM wykonane: {len(y_pred_llm)}")
    print(f"Unikalne predykcje: {np.unique(y_pred_llm, return_counts=True)}")
    
else:
    print("API Groq niedostępne. Ustaw zmienną środowiskową GROQ_API_KEY.")
    print("Aby uzyskać darmowy klucz: https://console.groq.com/keys")
    y_pred_llm = None

## 5. Ewaluacja i Porównanie

In [ ]:
# Funkcja do obliczania metryk
def calculate_metrics(y_true, y_pred, y_prob=None, name="Model"):
    """Oblicza wszystkie wymagane metryki."""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    
    # ROC-AUC (wymaga prawdopodobieństw)
    if y_prob is not None:
        y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
        try:
            auc = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")
        except:
            auc = None
    else:
        auc = None
    
    return {
        "model": name,
        "accuracy": acc,
        "macro_f1": f1,
        "roc_auc": auc,
        "predictions": y_pred
    }

In [ ]:
# Oblicz metryki dla wszystkich modeli
results = []

# Baseline
results.append(calculate_metrics(y_test, y_pred_baseline, y_prob_baseline, "TF-IDF + LogReg"))

# XGBoost
results.append(calculate_metrics(y_test, y_pred_xgb, y_prob_xgb, "XGBoost"))

# BERT
results.append(calculate_metrics(y_test, y_pred_bert, None, "BERT (128)"))

# LLM
if y_pred_llm is not None:
    results.append(calculate_metrics(y_test, y_pred_llm, None, f"LLM ({LLM_MODEL})"))

print("Metryki obliczone dla wszystkich modeli")

In [ ]:
# Tabela porównawcza
df_results = pd.DataFrame([{
    "Model": r["model"],
    "Accuracy": f"{r['accuracy']:.4f}",
    "Macro-F1 ⭐": f"{r['macro_f1']:.4f}",
    "ROC-AUC": f"{r['roc_auc']:.4f}" if r['roc_auc'] else "N/A"
} for r in results])

# Sortuj po Macro-F1 (główna metryka)
df_results_sorted = df_results.copy()
df_results_sorted["_f1_numeric"] = [float(r["macro_f1"]) for r in results]
df_results_sorted = df_results_sorted.sort_values("_f1_numeric", ascending=False).drop(columns=["_f1_numeric"])

print("\n" + "="*60)
print("           PORÓWNANIE MODELI (sortowane po Macro-F1)")
print("="*60)
display(df_results_sorted.reset_index(drop=True))

In [ ]:
# Wizualizacja: Wykres słupkowy
fig, ax = plt.subplots(figsize=(10, 6))

models = [r["model"] for r in results]
f1_scores = [r["macro_f1"] for r in results]
accuracies = [r["accuracy"] for r in results]

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, f1_scores, width, label='Macro-F1', color='steelblue')
bars2 = ax.bar(x + width/2, accuracies, width, label='Accuracy', color='coral')

ax.set_xlabel('Model')
ax.set_ylabel('Wynik')
ax.set_title('Porównanie Modeli: Detekcja Mowy Nienawiści')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1)

# Dodaj etykiety wartości
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(REPORTS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Macierze Pomyłek

In [ ]:
# Wyświetl macierze pomyłek dla wszystkich modeli
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 5))

if n_models == 1:
    axes = [axes]

for ax, r in zip(axes, results):
    cm = confusion_matrix(y_test, r["predictions"], normalize='true')
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format=".2f")
    ax.set_title(f"{r['model']}\nMacro-F1: {r['macro_f1']:.4f}")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Analiza Błędów (10 przypadków)

In [ ]:
# Znajdź błędne klasyfikacje
errors = []

for i, (text, true_label) in enumerate(zip(X_test, y_test)):
    preds = {
        "Baseline": int(y_pred_baseline[i]),
        "XGBoost": int(y_pred_xgb[i]),
        "BERT": int(y_pred_bert[i]),
    }
    if y_pred_llm is not None:
        preds["LLM"] = int(y_pred_llm[i])
    
    # Policz ile modeli się pomyliło
    wrong_count = sum(1 for p in preds.values() if p != true_label)
    
    if wrong_count > 0:
        errors.append({
            "index": i,
            "text": text[:300] + "..." if len(text) > 300 else text,
            "true_label": int(true_label),
            "true_class": CLASS_NAMES[true_label],
            "predictions": preds,
            "wrong_count": wrong_count
        })

# Sortuj po tym, ile modeli się pomyliło (najbardziej mylące przypadki)
errors_sorted = sorted(errors, key=lambda x: -x["wrong_count"])

print(f"Razem błędnych klasyfikacji: {len(errors)} / {len(y_test)}")
print(f"\nNajbardziej mylące przypadki (wszystkie lub większość modeli się pomyliła):")

In [ ]:
# Wyświetl 10 najbardziej mylących przypadków
error_analysis = []

for j, err in enumerate(errors_sorted[:10]):
    print(f"\n{'='*70}")
    print(f"Przypadek {j+1}: Prawda = {err['true_class']} ({err['true_label']})")
    print(f"Predykcje: {err['predictions']}")
    print(f"Błędnych modeli: {err['wrong_count']}")
    print(f"Tekst: {err['text']}")
    
    error_analysis.append({
        "case": j+1,
        "true_label": err['true_class'],
        "predictions": str(err['predictions']),
        "text_preview": err['text'][:100] + "...",
        "analysis": ""  # Do wypełnienia ręcznie
    })

# Zapisz analizę błędów
df_errors = pd.DataFrame(error_analysis)
df_errors.to_csv(REPORTS_DIR / "error_analysis.csv", index=False)
print(f"\n\nZapisano analizę błędów do {REPORTS_DIR / 'error_analysis.csv'}")

## 8. Zapis Końcowych Wyników

In [ ]:
# Zapisz wszystkie metryki do JSON
final_results = {
    "task": "3-class toxicity classification",
    "dataset": "Jigsaw Toxic Comment (Wikipedia)",
    "test_sample_size": len(y_test),
    "seed": SEED,
    "class_names": CLASS_NAMES,
    "models": [{
        "name": r["model"],
        "accuracy": float(r["accuracy"]),
        "macro_f1": float(r["macro_f1"]),
        "roc_auc": float(r["roc_auc"]) if r["roc_auc"] else None
    } for r in results]
}

with open(REPORTS_DIR / "final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print(f"Zapisano końcowe wyniki do {REPORTS_DIR / 'final_results.json'}")

In [ ]:
# Podsumowanie końcowe
print("\n" + "="*60)
print("                    PODSUMOWANIE KOŃCOWE")
print("="*60)
print(f"Rozmiar próbki testowej: {len(y_test)}")
print(f"Rozkład klas: 0={sum(y_test==0)}, 1={sum(y_test==1)}, 2={sum(y_test==2)}")
print("\nWyniki (sortowane po Macro-F1):")
for r in sorted(results, key=lambda x: -x['macro_f1']):
    auc_str = f", AUC={r['roc_auc']:.4f}" if r['roc_auc'] else ""
    print(f"  {r['model']:25s} Acc={r['accuracy']:.4f}, F1={r['macro_f1']:.4f}{auc_str}")
print("\nZapisane pliki:")
print(f"  - {REPORTS_DIR / 'model_comparison.png'}")
print(f"  - {REPORTS_DIR / 'confusion_matrices.png'}")
print(f"  - {REPORTS_DIR / 'error_analysis.csv'}")
print(f"  - {REPORTS_DIR / 'final_results.json'}")
print("\n✅ Projekt zakończony pomyślnie!")